# 🤖 ROTBOTS — Full Pipeline
## Topic → Video in One Notebook

---

All 9 stages. Run cells top to bottom.

> **🤔** Every cell is an automated decision. What does the machine choose?

---

In [ ]:
# SETUP — run once
!pip install -q requests beautifulsoup4 pymupdf edge-tts fal-client yt-dlp faster-whisper
import json, re, random, requests, subprocess, shutil, os, time as _time, threading, edge_tts
from pathlib import Path
from bs4 import BeautifulSoup
from IPython.display import display, Markdown, HTML, Audio, Video
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
BASE_DIR.mkdir(parents=True, exist_ok=True)
TEMP = Path('/content/temp_assembly'); TEMP.mkdir(exist_ok=True)
def progress_bar(c,t,l='',w=30):
    p=c/t if t>0 else 0;f=int(w*p);return f'<div style="font-family:monospace;font-size:14px;padding:2px 0;">{"█"*f}{"░"*(w-f)} {c}/{t} {l} ({p:.0%})</div>'
def progress_html(title,c,t,l='',d='',color='#4ecca3'):
    return f'<div style="background:#1a1a2e;padding:12px;border-radius:8px;color:#eaeaea;font-family:monospace;"><div style="font-size:16px;font-weight:bold;color:{color};">{title}</div>{progress_bar(c,t,l)}' + (f'<div style="color:#a0a0a0;font-size:12px;margin-top:4px;">{d}</div>' if d else '') + '</div>'
print('✅ Ready')

In [ ]:
# API KEYS
GROQ_API_KEY = ''  # https://console.groq.com/keys
FAL_API_KEY = ''   # https://fal.ai/dashboard/keys
LLM_MODEL='llama-3.3-70b-versatile';LLM_TEMPERATURE=0.8;LLM_MAX_TOKENS=4096
GROQ_API_URL='https://api.groq.com/openai/v1/chat/completions'
if GROQ_API_KEY: print('✅ Groq')
else: print('⚠️ Paste GROQ_API_KEY')
if FAL_API_KEY: os.environ['FAL_KEY']=FAL_API_KEY; print('✅ fal.ai')
else: print('⚠️ Paste FAL_API_KEY')

---
# ━━ 01: Video Plan

In [ ]:
TOPIC = 'The history and ethics of AI-generated art'
SOURCES = ['https://en.wikipedia.org/wiki/Artificial_intelligence_art']
TOTAL_VIDEO_LENGTH = 60
SECONDS_PER_SCENE = 5
ARCHIVE_RATIO = 0.0
UPLOAD_RATIO = 0.0
ENABLE_CREDITS = True
ENABLE_SUBTITLES = False
ENABLE_MUSIC = False
ENABLE_EFFECTS = True
SESSION_NAME = ''

In [ ]:
# CALCULATE
AI_RATIO=max(0,1.0-ARCHIVE_RATIO-UPLOAD_RATIO)
CREDITS_LENGTH=8 if ENABLE_CREDITS else 0
CONTENT_LENGTH=TOTAL_VIDEO_LENGTH-CREDITS_LENGTH
NARRATION_LENGTH=CONTENT_LENGTH
NUM_TOTAL_SCENES=max(2,int(CONTENT_LENGTH/SECONDS_PER_SCENE))
NUM_AI_SCENES=max(1,int(NUM_TOTAL_SCENES*AI_RATIO))
NUM_ARCHIVE_SCENES=int(NUM_TOTAL_SCENES*ARCHIVE_RATIO)
NUM_UPLOAD_SCENES=NUM_TOTAL_SCENES-NUM_AI_SCENES-NUM_ARCHIVE_SCENES
WORDS_PER_SCENE=SECONDS_PER_SCENE*2.5
NUM_CHAPTERS=max(1,min(3,NUM_TOTAL_SCENES//2))
SECTIONS_PER_CHAPTER=max(1,NUM_TOTAL_SCENES//NUM_CHAPTERS)
scene_types=[]
ai_p=0;ar_p=0;up_p=0
for i in range(NUM_TOTAL_SCENES):
    ai_d=(i+1)*AI_RATIO-ai_p;ar_d=(i+1)*ARCHIVE_RATIO-ar_p;up_d=(i+1)*UPLOAD_RATIO-up_p
    if ar_d>=ai_d and ar_d>=up_d and ar_p<NUM_ARCHIVE_SCENES: scene_types.append('archive');ar_p+=1
    elif up_d>=ai_d and up_p<NUM_UPLOAD_SCENES: scene_types.append('upload');up_p+=1
    else: scene_types.append('ai_generated');ai_p+=1
if not SESSION_NAME:
    words=re.sub(r'[^a-zA-Z0-9\s]','',TOPIC.lower()).split()
    SESSION_NAME='-'.join(words[:4])
SESSION_DIR=BASE_DIR/SESSION_NAME
SESSION_DIR.mkdir(parents=True,exist_ok=True)
(SESSION_DIR/'videos').mkdir(exist_ok=True)
(SESSION_DIR/'audio').mkdir(exist_ok=True)
VIDEOS_DIR=SESSION_DIR/'videos';AUDIO_DIR=SESSION_DIR/'audio'
plan={'topic':TOPIC,'sources':SOURCES,'session_name':SESSION_NAME,'total_video_length':TOTAL_VIDEO_LENGTH,'seconds_per_scene':SECONDS_PER_SCENE,'content_length':CONTENT_LENGTH,'credits_length':CREDITS_LENGTH,'narration_length':NARRATION_LENGTH,'ai_ratio':AI_RATIO,'archive_ratio':ARCHIVE_RATIO,'upload_ratio':UPLOAD_RATIO,'num_total_scenes':NUM_TOTAL_SCENES,'num_ai_scenes':NUM_AI_SCENES,'num_archive_scenes':NUM_ARCHIVE_SCENES,'num_upload_scenes':NUM_UPLOAD_SCENES,'scene_types':scene_types,'enable_credits':ENABLE_CREDITS,'enable_subtitles':ENABLE_SUBTITLES,'enable_music':ENABLE_MUSIC,'enable_effects':ENABLE_EFFECTS}
with open(SESSION_DIR/'video_plan.json','w') as f: json.dump(plan,f,indent=2)
with open(SESSION_DIR/'session_info.json','w') as f: json.dump({'name':SESSION_NAME,'topic':TOPIC},f,indent=2)
print(f'📂 {SESSION_NAME}: {TOTAL_VIDEO_LENGTH}s = {CONTENT_LENGTH}s + {CREDITS_LENGTH}s credits')
print(f'   🤖{NUM_AI_SCENES} AI 🏛️{NUM_ARCHIVE_SCENES} archive 📂{NUM_UPLOAD_SCENES} upload')
icons=['🤖' if t=='ai_generated' else '🏛️' if t=='archive' else '📂' for t in scene_types]
print(f'   {" → ".join(icons)}')

---
# ━━ 02: Script Writer

In [ ]:
# HELPERS
def query_llm(prompt,system_prompt=None,temperature=None):
    messages=[]
    if system_prompt: messages.append({'role':'system','content':system_prompt})
    messages.append({'role':'user','content':prompt})
    r=requests.post(GROQ_API_URL,headers={'Authorization':f'Bearer {GROQ_API_KEY}','Content-Type':'application/json'},json={'model':LLM_MODEL,'messages':messages,'temperature':temperature or LLM_TEMPERATURE,'max_tokens':LLM_MAX_TOKENS},timeout=120)
    r.raise_for_status();c=r.json()['choices'][0]['message']['content']
    if '<think>' in c and '</think>' in c: c=c.split('</think>')[-1].strip()
    return c
def parse_json_response(r):
    r=re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]','',r.strip())
    if r.startswith('```'): lines=r.split('\n');r='\n'.join(lines[1:-1] if lines[-1].strip()=='```' else lines[1:])
    r=r.strip()
    if not r.startswith('[') and not r.startswith('{'):
        for ch in ['{','[']: idx=r.find(ch)
        if idx!=-1: r=r[idx:]
    for end in ['}',']']:
        li=r.rfind(end)
        if li!=-1:
            t=r[:li+1]
            for text in [t,re.sub(r',\s*([}\]])',r'\1',t)]:
                try: return json.loads(text,strict=False)
                except: pass
    return json.loads(re.sub(r',\s*([}\]])',r'\1',r),strict=False)
def fetch_website_text(url,max_chars=10000):
    r=requests.get(url.strip(),headers={'User-Agent':'Mozilla/5.0'},timeout=30);r.raise_for_status()
    soup=BeautifulSoup(r.text,'html.parser')
    for el in soup(['script','style','nav','header','footer','aside','form']): el.decompose()
    main=None
    for sel in ['article','main','[role="main"]','.content','#content']:
        main=soup.select_one(sel)
        if main: break
    text=main.get_text(separator=' ',strip=True) if main else (soup.find('body') or soup).get_text(separator=' ',strip=True)
    return re.sub(r'\s+',' ',text).strip()[:max_chars]
print('✅ Helpers')

In [ ]:
# SCRAPE + SUMMARIZE
prog=display(HTML(''),display_id=True);source_texts=[];t0=_time.time()
for i,source in enumerate(SOURCES):
    prog.update(HTML(progress_html(f'📥 Source {i+1}/{len(SOURCES)}',i,len(SOURCES)*2,'steps',source[:60])))
    try: source_texts.append({'source':source[:100],'text':fetch_website_text(source) if source.startswith('http') else source})
    except Exception as e: print(f'   ❌ {e}')
summaries=[]
for i,src in enumerate(source_texts):
    prog.update(HTML(progress_html(f'🧠 Summarizing {i+1}/{len(source_texts)}',len(SOURCES)+i,len(SOURCES)*2,'steps')))
    summaries.append({'source':src['source'],'summary':query_llm(f'Summarize in 2-3 paragraphs for video essay about "{TOPIC}":\n\n{src["text"][:6000]}','Be concise.')})
with open(SESSION_DIR/'summaries.json','w') as f: json.dump({'topic':TOPIC,'sources':summaries},f,indent=2)
prog.update(HTML(progress_html(f'✅ {len(summaries)} sources ({_time.time()-t0:.1f}s)',len(SOURCES)*2,len(SOURCES)*2,'steps')))

In [ ]:
# OUTLINE + CHAPTERS
prog=display(HTML(''),display_id=True);t0=_time.time();total_llm=1+NUM_CHAPTERS
prog.update(HTML(progress_html('🧠 Outline...',0,total_llm,'LLM')))
st='\n\n'.join([f'SOURCE: {s["source"]}\n{s["summary"]}' for s in summaries])
for attempt in range(3):
    try: outline=parse_json_response(query_llm(f'Essay outline for {NARRATION_LENGTH}s video: "{TOPIC}"\n\nRESEARCH:\n{st}\n\nExactly {NUM_CHAPTERS} chapters. JSON only: {{"title":"...","thesis":"...","chapters":[{{"chapter":1,"title":"...","summary":"...","key_points":["..."]}}]}}','Expert essay writer.')); break
    except: pass
if len(outline.get('chapters',[]))>NUM_CHAPTERS: outline['chapters']=outline['chapters'][:NUM_CHAPTERS]
for i,ch in enumerate(outline['chapters']):
    prog.update(HTML(progress_html(f'✍️ Ch {ch["chapter"]}: {ch["title"]}',i+1,total_llm,'LLM')))
    try:
        sections=parse_json_response(query_llm(f'Write {SECTIONS_PER_CHAPTER} sections. MAX {int(WORDS_PER_SCENE)} words each.\nTOPIC: {TOPIC}\nCHAPTER: {ch["title"]}\nJSON: [{{"section":1,"narration":"...","visual_direction":"...","mood":"..."}}]',f'Scriptwriter. MAX {int(WORDS_PER_SCENE)} words per section.'))
        if isinstance(sections,dict):
            for v in sections.values():
                if isinstance(v,list): sections=v; break
            else: sections=[sections]
    except: sections=[{'section':1,'narration':ch['title'],'visual_direction':'','mood':'neutral'}]
    outline['chapters'][i]['sections']=sections[:SECTIONS_PER_CHAPTER]
outline['credits']={'sources':[s['source'] for s in summaries]}
with open(SESSION_DIR/'essay_script.json','w') as f: json.dump(outline,f,indent=2)
tw=sum(len(s.get('narration','').split()) for c in outline['chapters'] for s in c.get('sections',[]))
prog.update(HTML(progress_html(f'✅ "{outline.get("title","")}" — {tw}w ≈ {tw/2.5:.0f}s',total_llm,total_llm,'LLM')))

In [ ]:
# STORYBOARD + STYLES
STYLE_ARCS={'calm_to_dramatic':{'early':['documentary','nature'],'middle':['news_report','sports_commentary'],'late':['action_movie','horror']}}
STYLES={'documentary':{'vk':'cinematic'},'nature':{'vk':'golden hour'},'news_report':{'vk':'professional'},'sports_commentary':{'vk':'dynamic'},'action_movie':{'vk':'dramatic'},'horror':{'vk':'dark, fog'}}
arc=STYLE_ARCS['calm_to_dramatic']
all_sec=[]
for ch in outline.get('chapters',[]):
    for sec in ch.get('sections',[]): all_sec.append({**sec,'chapter_title':ch['title'],'chapter':ch.get('chapter',0)})
scenes=[];sn=1;si=0
for stype in scene_types:
    sec=all_sec[si] if si<len(all_sec) else {'narration':'','visual_direction':'','mood':'neutral','chapter_title':''}
    scenes.append({'scene':sn,'scene_type':stype,'title':f'{sec.get("chapter_title","")} - Part {sec.get("section",si+1)}','narration':sec.get('narration',''),'visual_direction':sec.get('visual_direction',''),'mood':sec.get('mood',''),'duration':SECONDS_PER_SCENE})
    sn+=1;si+=1
ai_sc=[s for s in scenes if s['scene_type']=='ai_generated'];tal=len(ai_sc)
ee=max(1,int(tal*0.25));ls=max(ee+1,int(tal*0.75));last=None
for i,sc in enumerate(ai_sc):
    pool=arc.get('early' if i<ee else 'late' if i>=ls else 'middle',['documentary'])
    av=[s for s in pool if s!=last] or pool;st=random.choice(av)
    sc['assigned_style']=st;sc['visual_keywords']=STYLES.get(st,{}).get('vk','');last=st
with open(SESSION_DIR/'storyboard.json','w') as f: json.dump(scenes,f,indent=2)
for s in scenes:
    icon='🤖' if s['scene_type']=='ai_generated' else '🏛️' if s['scene_type']=='archive' else '📂'
    print(f'   {icon} Scene {s["scene"]}: {s["title"]} ({s["scene_type"]})')

In [ ]:
# T2V PROMPTS (AI scenes only)
OPENINGS=['Start with SHOT TYPE','Start with ACTION','Start with ENVIRONMENT']
prompts=[];prog=display(HTML(''),display_id=True);t0=_time.time()
for idx,sc in enumerate([s for s in scenes if s['scene_type']=='ai_generated']):
    st=sc.get('assigned_style','none')
    prog.update(HTML(progress_html(f'🎥 Scene {sc["scene"]}',idx,NUM_AI_SCENES,'prompts',f'Style: {st}')))
    t2v=query_llm(f'T2V prompt for {SECONDS_PER_SCENE}s:\nVisual: {sc.get("visual_direction","")}\nMood: {sc.get("mood","")}\n{random.choice(OPENINGS)}\nOutput ONLY prompt.',f'T2V expert. ONE paragraph. Style:{st} Visual:{sc.get("visual_keywords","")}')
    prompts.append({'scene':sc['scene'],'title':sc['title'],'t2v_prompt':t2v.strip().strip('"'),'style':st,'narration':sc.get('narration',''),'duration':SECONDS_PER_SCENE})
with open(SESSION_DIR/'prompts.json','w') as f: json.dump(prompts,f,indent=2)
prog.update(HTML(progress_html(f'✅ {len(prompts)} prompts ({_time.time()-t0:.1f}s)',NUM_AI_SCENES,NUM_AI_SCENES,'prompts')))

---
# ━━ 05: Effects

In [ ]:
# ASSIGN EFFECTS
EFFECT_MODE='random';EFFECT_INTENSITY=0.7
if ENABLE_EFFECTS:
    enames=['film_grain','vhs_artifacts','celluloid_scratches','sepia_tone','bw_transition','color_grade_warm','color_grade_cool','vignette','flicker','desaturate']
    for p in prompts: p['ffmpeg_effects']=[random.choice(enames)];p['effect_intensity']=EFFECT_INTENSITY
    with open(SESSION_DIR/'prompts.json','w') as f: json.dump(prompts,f,indent=2)
    for p in prompts: print(f'   Scene {p["scene"]}: {p["ffmpeg_effects"][0]}')
else: print('🎨 Effects disabled')

---
# ━━ 06: The Voice

In [ ]:
# TTS
VOICES={'documentary':'en-US-GuyNeural','female_us':'en-US-AriaNeural','male_uk':'en-GB-RyanNeural'}
CHOSEN_VOICE='documentary';voice_name=VOICES[CHOSEN_VOICE]
async def gen_tts(text,path,voice,rate='+0%'):
    await edge_tts.Communicate(text,voice,rate=rate).save(str(path))
def get_dur(path):
    try: return float(subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(path)],capture_output=True,text=True,timeout=10).stdout.strip())
    except: return 5.0
prog=display(HTML(''),display_id=True);audio_files=[];t0=_time.time()
narrations=[{'scene':s['scene'],'narration':s['narration']} for s in scenes if s.get('narration')]
for idx,n in enumerate(narrations):
    out=AUDIO_DIR/f'narration_{n["scene"]:03d}.mp3'
    el=_time.time()-t0;sp=idx/el if el>0 else 0;eta=(len(narrations)-idx)/sp if sp>0 else 0
    prog.update(HTML(progress_html(f'🎙️ Scene {n["scene"]}',idx,len(narrations),'narrations',f'⏱️ {el:.0f}s · ~{eta:.0f}s left')))
    try:
        await gen_tts(n['narration'],out,voice_name)
        audio_files.append({'scene':n['scene'],'path':str(out),'duration':get_dur(out),'text':n['narration']})
    except Exception as e: print(f'   ❌ {e}')
with open(SESSION_DIR/'audio_manifest.json','w') as f: json.dump({'voice':voice_name,'files':audio_files},f,indent=2)
td=sum(a['duration'] for a in audio_files)
prog.update(HTML(progress_html(f'✅ {len(audio_files)} narrations, {td:.1f}s',len(narrations),len(narrations),'narrations')))

---
# ━━ 07: AI Video Generation

In [ ]:
# GENERATE
import fal_client
MODELS={'wan-t2v':{'endpoint':'fal-ai/wan-t2v','ds':5},'ltx-video':{'endpoint':'fal-ai/ltx-video/v2.1','ds':5}}
CHOSEN_MODEL='wan-t2v';model=MODELS[CHOSEN_MODEL]
prog=display(HTML(''),display_id=True);generated_videos=[];t0=_time.time();_stop=False
def _timer(ph,si,done,total,ts):
    sp=['⠋','⠙','⠹','⠸','⠼','⠴','⠦','⠧','⠇','⠏'];tick=0
    while not _stop:
        el=_time.time()-ts;ce=_time.time()-si.get('cs',ts);avg=el/done[0] if done[0]>0 else 0;eta=(total-done[0]-1)*avg+max(0,avg-ce) if avg>0 else 0
        s=sp[tick%len(sp)];p=done[0]/total if total>0 else 0;fl=int(30*p)
        try:ph.update(HTML(f'<div style="background:#1a1a2e;padding:12px;border-radius:8px;color:#eaeaea;font-family:monospace;"><div style="font-size:16px;font-weight:bold;color:#e94560;">{s} Scene {si.get("sn","?")}: {si.get("title","")}</div><div style="font-size:14px;">{"█"*fl}{"░"*(30-fl)} {done[0]}/{total} ({p:.0%})</div><div style="color:#a0a0a0;font-size:12px;">⏱️ {el:.0f}s · clip: {ce:.0f}s · ~{eta:.0f}s left</div></div>'))
        except:pass
        tick+=1;_time.sleep(1)
for i,pd in enumerate(prompts):
    sn=pd['scene'];si={'sn':sn,'title':pd['title'],'cs':_time.time()};done=[len(generated_videos)]
    _stop=False;t=threading.Thread(target=_timer,args=(prog,si,done,len(prompts),t0),daemon=True);t.start()
    try:
        inp={'prompt':pd['t2v_prompt'],'aspect_ratio':'16:9','enable_prompt_expansion':False}
        result=fal_client.subscribe(model['endpoint'],arguments=inp);url=None
        if isinstance(result,dict):
            if 'video' in result and isinstance(result['video'],dict):url=result['video'].get('url')
            elif 'video' in result:url=result['video']
            elif 'url' in result:url=result['url']
        if url:
            vp=VIDEOS_DIR/f'scene_{sn:03d}.mp4'
            with open(vp,'wb') as f:f.write(requests.get(url,timeout=120).content)
            generated_videos.append({'scene':sn,'path':str(vp),'duration':pd.get('duration',5),'source':'generated'})
    except:pass
    finally:_stop=True;t.join(timeout=2)
prog.update(HTML(f'<div style="background:#1a1a2e;padding:12px;border-radius:8px;color:#eaeaea;font-family:monospace;"><div style="font-size:16px;font-weight:bold;color:#4ecca3;">✅ {len(generated_videos)}/{len(prompts)} videos in {_time.time()-t0:.0f}s</div></div>'))
with open(SESSION_DIR/'video_manifest.json','w') as f:json.dump({'model':CHOSEN_MODEL,'videos':generated_videos},f,indent=2)

---
# ━━ 09: Assemble

In [ ]:
# HELPERS
def dur(p):
    try: return float(subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(p)],capture_output=True,text=True,timeout=10).stdout.strip())
    except: return 5.0
def ff(cmd,desc=''):
    if desc: print(f'   {desc}...',end=' ',flush=True)
    r=subprocess.run(cmd,capture_output=True,text=True,timeout=600)
    if r.returncode==0:
        if desc: print('✅')
        return True
    if desc: print('❌')
    print(f'   {r.stderr[-300:]}'); return False
def build_effect_filter(name,intensity=0.7):
    i=max(0.0,min(1.0,intensity))
    if name=='film_grain':return f'noise=alls={int(12*i)}:allf=t'
    if name=='vhs_artifacts':return f'noise=alls={int(30*i)}:allf=t,rgbashift=rh={int(2*i)}:bh={int(-2*i)}'
    if name=='sepia_tone':inv=1-i;return f'colorchannelmixer={inv+i*0.393:.3f}:{i*0.769:.3f}:{i*0.189:.3f}:0:{i*0.349:.3f}:{inv+i*0.686:.3f}:{i*0.168:.3f}:0:{i*0.272:.3f}:{i*0.534:.3f}:{inv+i*0.131:.3f}:0'
    if name=='bw_transition':inv=1-i;return f'colorchannelmixer={inv+i*0.3:.3f}:{i*0.4:.3f}:{i*0.3:.3f}:0:{i*0.3:.3f}:{inv+i*0.4:.3f}:{i*0.3:.3f}:0:{i*0.3:.3f}:{i*0.4:.3f}:{inv+i*0.3:.3f}:0'
    if name=='color_grade_warm':return f'eq=saturation={1+0.1*i:.3f}:brightness={0.02*i:.3f}'
    if name=='color_grade_cool':return f'eq=saturation={1-0.1*i:.3f},colorbalance=bs={0.12*i:.3f}'
    if name=='vignette':return f'vignette=PI/4*{i:.3f}'
    if name=='desaturate':return f'eq=saturation={0.5+0.5*(1-i):.3f}'
    if name=='celluloid_scratches':return f'noise=c0s={int(15*i)}:c0f=t'
    if name=='flicker':return f"noise=alls={int(8*i)}:allf=t"
    return None

In [ ]:
# BUILD SEQUENCE
effects_map={int(p['scene']):p for p in prompts if p.get('ffmpeg_effects')}
sub_file=SESSION_DIR/'subtitles.ass'
music_path=None;credits_path=None
print(f'🔧 Building {len(scenes)} scenes...')
norm=[];arc_idx=0;upl_idx=0
archive_clips=[]
if (SESSION_DIR/'archive_clips.json').exists():
    with open(SESSION_DIR/'archive_clips.json') as f: archive_clips=json.load(f).get('clips',[])
upload_clips=[]
if (SESSION_DIR/'upload_clips.json').exists():
    with open(SESSION_DIR/'upload_clips.json') as f: upload_clips=json.load(f).get('clips',[])
for sc in scenes:
    sn=sc['scene'];stype=sc['scene_type']
    icon='🤖' if stype=='ai_generated' else '🏛️' if stype=='archive' else '📂'
    out=TEMP/f'scene_{sn:03d}.mp4';src=None
    if stype=='ai_generated':
        c=VIDEOS_DIR/f'scene_{sn:03d}.mp4'
        if c.exists():src=c
        else:print(f'   {icon} Scene {sn}: ❌ missing');continue
    elif stype=='archive':
        if arc_idx<len(archive_clips):src=Path(archive_clips[arc_idx]['path']);arc_idx+=1
        if not src or not src.exists():print(f'   {icon} Scene {sn}: ❌ missing');continue
    elif stype=='upload':
        if upl_idx<len(upload_clips):src=Path(upload_clips[upl_idx]['path']);upl_idx+=1
        if not src or not src.exists():continue
    vf='scale=1280:720:force_original_aspect_ratio=decrease,pad=1280:720:(ow-iw)/2:(oh-ih)/2:black'
    if stype=='ai_generated' and sn in effects_map:
        for eff in effects_map[sn].get('ffmpeg_effects',[]):
            ef=build_effect_filter(eff,effects_map[sn].get('effect_intensity',0.7))
            if ef:vf+=','+ef
    if ff(['ffmpeg','-y','-i',str(src),'-vf',vf,'-r','24','-pix_fmt','yuv420p','-c:v','libx264','-preset','fast','-crf','23','-an','-t',str(sc.get('duration',5)),str(out)],f'{icon} Scene {sn}'):
        norm.append(out)
print(f'✅ {len(norm)} scenes')

In [ ]:
# CREDITS + CONCAT + AUDIO (PADDED) + SUBS → FINAL
if ENABLE_CREDITS:
    essay=None
    if (SESSION_DIR/'essay_script.json').exists():
        with open(SESSION_DIR/'essay_script.json') as f: essay=json.load(f)
    title=essay.get('title','Untitled') if essay else 'Untitled'
    lines=[title,'','Generated by ROTBOTS','','— Sources —']+[s[:60] for s in (essay.get('credits',{}).get('sources',[]) if essay else [])]+['','LI-MA TDA 2026']
    D=8;LH=42;spd=(720+len(lines)*LH)/D
    flt=[f"drawtext=text='{l.replace(chr(39),chr(8217)).replace(chr(58),chr(92)+chr(58))}':fontsize=28:fontcolor=white:x=(w-text_w)/2:y=h+{i*LH}-{spd:.1f}*t" for i,l in enumerate(lines) if l]
    credits_path=TEMP/'credits.mp4'
    ff(['ffmpeg','-y','-f','lavfi','-i',f'color=c=black:s=1280x720:d={D}:r=24','-vf',','.join(flt),'-pix_fmt','yuv420p','-c:v','libx264','-preset','fast',str(credits_path)],'Credits')

# Concat
cf=TEMP/'concat.txt'
with open(cf,'w') as f:
    for v in norm: f.write(f"file '{v}'\n")
    if credits_path and credits_path.exists(): f.write(f"file '{credits_path}'\n")
concat_out=TEMP/'concatenated.mp4'
ff(['ffmpeg','-y','-f','concat','-safe','0','-i',str(cf),'-c','copy',str(concat_out)],'Concat')
video_duration=dur(concat_out)
print(f'   Video: {video_duration:.1f}s')

# Audio with padding (prevents credits from being cut off)
audio_out=TEMP/'with_audio.mp4'
audio_files_sorted=sorted(AUDIO_DIR.glob('narration_*.mp3')) if AUDIO_DIR.exists() else []
if audio_files_sorted:
    acf=TEMP/'ac.txt'
    with open(acf,'w') as f:
        for a in audio_files_sorted: f.write(f"file '{a}'\n")
    narr=TEMP/'narration.mp3'
    ff(['ffmpeg','-y','-f','concat','-safe','0','-i',str(acf),'-c','copy',str(narr)],'Concat narration')
    # Pad with silence to match video length
    narr_padded=TEMP/'narration_padded.mp3'
    ff(['ffmpeg','-y','-i',str(narr),'-af',f'apad=whole_dur={video_duration}','-c:a','libmp3lame','-b:a','128k',str(narr_padded)],'Pad audio')
    ff(['ffmpeg','-y','-i',str(concat_out),'-i',str(narr_padded),'-c:v','copy','-c:a','aac','-b:a','192k','-map','0:v:0','-map','1:a:0',str(audio_out)],'Merge audio')
else: shutil.copy2(str(concat_out),str(audio_out))

# Subtitles
final=SESSION_DIR/'final_video.mp4'
if ENABLE_SUBTITLES and sub_file.exists():
    la=TEMP/'subtitles.ass';shutil.copy2(str(sub_file),str(la))
    if not ff(['ffmpeg','-y','-i',str(audio_out),'-vf',f'ass={str(la)}','-c:v','libx264','-preset','fast','-crf','23','-c:a','copy',str(final)],'Burn subs'):
        shutil.copy2(str(audio_out),str(final))
else: shutil.copy2(str(audio_out),str(final))
if final.exists(): print(f'\n✅ Final: {dur(final):.1f}s, {final.stat().st_size/(1024*1024):.1f}MB')

---
## 🎬 Watch!

In [ ]:
essay=None
if (SESSION_DIR/'essay_script.json').exists():
    with open(SESSION_DIR/'essay_script.json') as f: essay=json.load(f)
title=essay.get('title','Untitled') if essay else 'Untitled'
if final.exists():
    display(Markdown(f'# 🎬 {title}\n*Generated by ROTBOTS*\n\n---'))
    try: display(Video(str(final),embed=True,width=720))
    except: print(f'Drive: rotbots_workshop/{SESSION_NAME}/final_video.mp4')

---
# 🏁 Done

**Every step: automated decisions.** What does that mean?

---
*ROTBOTS — LI-MA TDA 2026, Amsterdam*